# N-gram language model

In [53]:
import numpy as np
import math

## Phase 1: Dataset & Tokenization


In [54]:
corpus = [
    "I love  machine  learning",
    "I love deep learning",
    "I enjoy natural language processing",
    "Machine learning is fun",
    "Deep learning is powerful"
]
sentence = "I   love   machine learning ! !"

In [55]:
# sentence tokenization 
def tokenize_sentence(sentence: str) ->list[str]:
    """
    Convert a sentence into a list of lowercase tokens and add the
    start (<s>) and end (</s>) tokens.

    Args:
        sentence (str): The input sentence
    
    Returns:
        list[str]: Tokenized sentence
    """
    puncs = ".?!,"
    sentence = sentence.lower()
    for punc in puncs:
        sentence = sentence.replace(punc,'')
    tokenized = sentence.split()

    return ["<s>"] + tokenized + ["</s>"]


In [56]:
# corpus tokenization 
def tokenize_corpus(corpus: list[str]) ->list[list[str]]:
    """
    Convert corpus into a list of tokenized sentences.

    Args:
        corpus (list[str]): The input corpus.
    
    Returns:
        list[list[str]]: The tokenized corpus.
    """


    tokenized = []    
    for sentence in corpus:        
        tokenized.append(tokenize_sentence(sentence))

    return tokenized 
    #  Or return [tokenize_sentence(sentence) for sentence in corpus]

In [57]:
tokenized_corpus = tokenize_corpus(corpus)
tokenized_corpus

[['<s>', 'i', 'love', 'machine', 'learning', '</s>'],
 ['<s>', 'i', 'love', 'deep', 'learning', '</s>'],
 ['<s>', 'i', 'enjoy', 'natural', 'language', 'processing', '</s>'],
 ['<s>', 'machine', 'learning', 'is', 'fun', '</s>'],
 ['<s>', 'deep', 'learning', 'is', 'powerful', '</s>']]

## Phase 2: vocabulary 

vocabulary is the set of all unique tokens that appear in the corpus.

In our implementation, we use Python's `set` because it automatically prevents duplicates entries.  

In [ ]:
def build_vocabulary (tokenized_corpus: list[list[str]])->set[str]:
    """
    Extract the unique words from a tokenized corpus.

    Args:
        tokenized_corpus (list[list[str]]): The tokenized corpus.
    
    Returns:
        set(str): The vocabulary.
    """    
    
    vocab = set()

    for sentence in tokenized_corpus:
        for word in sentence:
            vocab.add(word)

    return vocab

## Phase 3: Unigram

In [ ]:
def count_unigrams(tokenized_corpus : list[list[str]])->dict[str, int]:
    """
    Count the number of appearance of each unigram(single word).

    Args:
        tokenized_corpus (list[list[str]]): The tokenized corpus.
    
    Returns:
        dict[str, int]: count(int)}: A mapping from each word to its frequency.
    """        
    
    count  = {}
    for sentence in tokenized_corpus:
        for word in sentence:
            if word in count:
                count[word] +=1
            else:
                count[word] = 1
    return (count)

In [96]:
counts_unigrams = count_unigrams(tokenized_corpus)
print(counts_unigrams)

{'<s>': 5, 'i': 3, 'love': 2, 'machine': 2, 'learning': 4, '</s>': 5, 'deep': 2, 'enjoy': 1, 'natural': 1, 'language': 1, 'processing': 1, 'is': 2, 'fun': 1, 'powerful': 1}


In [61]:
def compute_unigram_probabilities(counted_unigrams:dict):
    total_words = sum(counted_unigrams.values())
    
    unigram_probabilities = {}
    
    for word, value in counted_unigrams.items():
        unigram_probabilities[word] = value/ total_words
    
    return (unigram_probabilities)

"""
Complexity

for vocabulary size V
we perform: sum(counted_unigrams.values()) 
which visits every unique word once.

So total complexity:
- Time: O(V)
- Space: O(V)
"""

'\nComplexity\n\nfor vocabulary size V\nwe perform: sum(counted_unigrams.values()) \nwhich visits every unique word once.\n\nSo total complexity:\n- Time: O(V)\n- Space: O(V)\n'

## Phase 4: N-grams function 

**Sliding window**: algorithm technique  used to process consecutive elements efficiently.  

For N-grams:
- window size = n.
- -step = 1.

Sentence:
i love machine learning

n = 2

Windows:

(i, love)  
(love, machine)  
(machine, learning)  

**Formula**:

$$
\text {Number of n-grams=L−n+1}
$$

where 
- L: length of sentence.
- $
n \leq {L}
$


In [62]:
def generate_ngrams(tokenized_corpus, n : int ):
    
    if n <= 0:
        raise ValueError("n must be greater than 0") 

    n_gram = []

    for sentence in tokenized_corpus:
        L = len(sentence)
        
        for i in range (L - n + 1):        

            n_gram.append(tuple(sentence [i:n+i]))
    
    return (n_gram)

""" 
Complexity:
- Time O(N)
- Space: O(number of generated n-grams)
"""


' \nComplexity:\n- Time O(N)\n- Space: O(number of generated n-grams)\n'

In [63]:
generated_ngrams = generate_ngrams(tokenized_corpus,2)
print(generated_ngrams)

[('<s>', 'i'), ('i', 'love'), ('love', 'machine'), ('machine', 'learning'), ('learning', '</s>'), ('<s>', 'i'), ('i', 'love'), ('love', 'deep'), ('deep', 'learning'), ('learning', '</s>'), ('<s>', 'i'), ('i', 'enjoy'), ('enjoy', 'natural'), ('natural', 'language'), ('language', 'processing'), ('processing', '</s>'), ('<s>', 'machine'), ('machine', 'learning'), ('learning', 'is'), ('is', 'fun'), ('fun', '</s>'), ('<s>', 'deep'), ('deep', 'learning'), ('learning', 'is'), ('is', 'powerful'), ('powerful', '</s>')]


In [64]:
def count_ngrams(generated_ngrams):
    count = {}
    for n_gram in generated_ngrams:
        if n_gram in count: # same as "if n_gram in count.keys()" dictionaries check their keys by default.
            count[n_gram] +=1
        else:
            count[n_gram] = 1 
    
    return count

""" 
Complexity:
- Time O(M) where M is the number of generated n-grams
- Space: O(U) where U is the number of unique n-grams
"""


' \nComplexity:\n- Time O(M) where M is the number of generated n-grams\n- Space: O(U) where U is the number of unique n-grams\n'

In [65]:
print(count_ngrams(generated_ngrams))

{('<s>', 'i'): 3, ('i', 'love'): 2, ('love', 'machine'): 1, ('machine', 'learning'): 2, ('learning', '</s>'): 2, ('love', 'deep'): 1, ('deep', 'learning'): 2, ('i', 'enjoy'): 1, ('enjoy', 'natural'): 1, ('natural', 'language'): 1, ('language', 'processing'): 1, ('processing', '</s>'): 1, ('<s>', 'machine'): 1, ('learning', 'is'): 2, ('is', 'fun'): 1, ('fun', '</s>'): 1, ('<s>', 'deep'): 1, ('is', 'powerful'): 1, ('powerful', '</s>'): 1}


## Phase 5: Bigrams


In [66]:
# Generate bigrams
bigrams = generate_ngrams(tokenized_corpus, 2)

In [67]:
# Count bigrams
counts_bigram = count_ngrams(bigrams)

In [68]:
def compute_bigram_probabilities(counts_bigram,counts_unigrams ):
    prob_bigram = {}
    
    for bigram,bigram_value in counts_bigram.items():
        unigram_value = counts_unigrams[bigram[0]]
        prob_bigram[bigram] = bigram_value/unigram_value
    return prob_bigram


In [69]:
print (compute_bigram_probabilities(counts_bigram,counts_unigrams ))

{('<s>', 'i'): 0.6, ('i', 'love'): 0.6666666666666666, ('love', 'machine'): 0.5, ('machine', 'learning'): 1.0, ('learning', '</s>'): 0.5, ('love', 'deep'): 0.5, ('deep', 'learning'): 1.0, ('i', 'enjoy'): 0.3333333333333333, ('enjoy', 'natural'): 1.0, ('natural', 'language'): 1.0, ('language', 'processing'): 1.0, ('processing', '</s>'): 1.0, ('<s>', 'machine'): 0.2, ('learning', 'is'): 0.5, ('is', 'fun'): 0.5, ('fun', '</s>'): 1.0, ('<s>', 'deep'): 0.2, ('is', 'powerful'): 0.5, ('powerful', '</s>'): 1.0}


## Phase 6: Sentence Scoring

Sentence Scoring can be used by both MLE and  laplace smoothed probabilities 

In [70]:
sentence = ["i", "love", "machine", "learning"]

tokenized_corpus = tokenize_corpus(corpus)

counts_unigrams = count_unigrams(tokenized_corpus)

bigrams = generate_ngrams(tokenized_corpus, 2)

counts_bigram = count_ngrams(bigrams)

bigram_probabilities = compute_bigram_probabilities(counts_bigram,counts_unigrams )

print(bigram_probabilities)


{('<s>', 'i'): 0.6, ('i', 'love'): 0.6666666666666666, ('love', 'machine'): 0.5, ('machine', 'learning'): 1.0, ('learning', '</s>'): 0.5, ('love', 'deep'): 0.5, ('deep', 'learning'): 1.0, ('i', 'enjoy'): 0.3333333333333333, ('enjoy', 'natural'): 1.0, ('natural', 'language'): 1.0, ('language', 'processing'): 1.0, ('processing', '</s>'): 1.0, ('<s>', 'machine'): 0.2, ('learning', 'is'): 0.5, ('is', 'fun'): 0.5, ('fun', '</s>'): 1.0, ('<s>', 'deep'): 0.2, ('is', 'powerful'): 0.5, ('powerful', '</s>'): 1.0}


In [71]:
def compute_sentence_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    prob = 1

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            prob = prob * bigram_probabilities[bigram]
        else:
            return 0
    return prob 
# we wont use compute_sentence_probability instead we are going to use compute_sentence_log_probability

In [72]:
sentence_probability = compute_sentence_probability(sentence,bigram_probabilities)
print(sentence_probability)

0.3333333333333333


In [73]:
def compute_sentence_log_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    log_prob  = 0

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            p = bigram_probabilities[bigram]
            log_prob  = log_prob  + np.log(p)
        else:
            return float("-inf") # we cant return 0 as log(0) = - inf so we use this: float("-inf")
    return log_prob

In [74]:
sentence2 =  ["i", "love", "powerful", "learning"]

In [75]:
sentence_probability = compute_sentence_probability(sentence2,bigram_probabilities)
print(sentence_probability)

0


In [76]:
sentence_log_probability = compute_sentence_log_probability(sentence2,bigram_probabilities)
print(sentence_log_probability)

-inf


## Phase 7: Smoothing

In [77]:
counts_unigrams = count_unigrams(tokenized_corpus)

counts_bigrams = count_ngrams(bigrams)

vocab = build_vocabulary(tokenized_corpus)
vocab_size = len(vocab) 


In [78]:
k = 1
prob_bigram = {}
for bigram,bigram_value in counts_bigrams.items():
    unigram_value = counts_unigrams[bigram[0]]
    prob_bigram[bigram] = (bigram_value + k) /(unigram_value + k * vocab_size)
    print (bigram)

('<s>', 'i')
('i', 'love')
('love', 'machine')
('machine', 'learning')
('learning', '</s>')
('love', 'deep')
('deep', 'learning')
('i', 'enjoy')
('enjoy', 'natural')
('natural', 'language')
('language', 'processing')
('processing', '</s>')
('<s>', 'machine')
('learning', 'is')
('is', 'fun')
('fun', '</s>')
('<s>', 'deep')
('is', 'powerful')
('powerful', '</s>')


In [79]:
def compute_bigram_probabilities_smoothed(counts_bigrams, counts_unigrams, vocab:set, k = 1 ):
    
    prob_bigram = {}
    vocab_size = len(vocab) 
    

    for previous_word in vocab:
        for current_word in vocab:
            bigram = (previous_word,current_word)
            unigram_value = counts_unigrams[previous_word]

            if bigram not in counts_bigrams: # the if condition is same as: bigram_value = counts_bigrams.get(bigram, 0) 
                bigram_value = 0
            else:
                bigram_value = counts_bigrams[bigram]
            
                
            prob_bigram[bigram] = (bigram_value + k) /(unigram_value + k * vocab_size)
    return (prob_bigram)


In [80]:
bigram_probabilities_smoothed = compute_bigram_probabilities_smoothed(counts_bigrams,counts_unigrams,vocab )
bigram_probabilities_smoothed

{('learning', 'learning'): 0.05555555555555555,
 ('learning', 'love'): 0.05555555555555555,
 ('learning', 'is'): 0.16666666666666666,
 ('learning', 'powerful'): 0.05555555555555555,
 ('learning', 'deep'): 0.05555555555555555,
 ('learning', 'fun'): 0.05555555555555555,
 ('learning', 'natural'): 0.05555555555555555,
 ('learning', '</s>'): 0.16666666666666666,
 ('learning', 'i'): 0.05555555555555555,
 ('learning', 'machine'): 0.05555555555555555,
 ('learning', 'processing'): 0.05555555555555555,
 ('learning', '<s>'): 0.05555555555555555,
 ('learning', 'enjoy'): 0.05555555555555555,
 ('learning', 'language'): 0.05555555555555555,
 ('love', 'learning'): 0.0625,
 ('love', 'love'): 0.0625,
 ('love', 'is'): 0.0625,
 ('love', 'powerful'): 0.0625,
 ('love', 'deep'): 0.125,
 ('love', 'fun'): 0.0625,
 ('love', 'natural'): 0.0625,
 ('love', '</s>'): 0.0625,
 ('love', 'i'): 0.0625,
 ('love', 'machine'): 0.125,
 ('love', 'processing'): 0.0625,
 ('love', '<s>'): 0.0625,
 ('love', 'enjoy'): 0.0625,
 ('

**Should the probabilities sum to 1?**  

Yes but they don't all sum to 1.  
Instead for each previous word the conditional distribution sums to 1

$$
\sum_{w_i \in V} P(w_i \mid w_{i-1}) = 1
$$

Example:
``` text
Vocabulary:
<s>
I
love
NLP
</s>
```
then 

```text
P(<s> | love)
P(I | love)
P(love | love)
P(NLP | love)
P(</s> | love)
````
must sum to 1

In [81]:
# test the sum of the probabilities for previous words
for previous_word in vocab:
    total = 0

    for current_word in vocab:
        total += bigram_probabilities_smoothed[(previous_word, current_word)]

    print(previous_word, total)

learning 1.0000000000000002
love 1.0
is 1.0
powerful 0.9999999999999999
deep 1.0
fun 0.9999999999999999
natural 0.9999999999999999
</s> 0.7368421052631577
i 1.0
machine 1.0
processing 0.9999999999999999
<s> 0.9999999999999998
enjoy 0.9999999999999999
language 0.9999999999999999


In [82]:
# note the total sum is not equal 1
sum(bigram_probabilities_smoothed.values())

13.736842105263134

```text
                Probability Model
                      │
        ┌─────────────┴─────────────┐
        │                           │
      MLE                     Laplace Smoothing
        │                           │
        └─────────────┬─────────────┘
                      │
compute_sentence_log_probability(...)
```

In [83]:
log_probability_smoothed = compute_sentence_log_probability(sentence2, bigram_probabilities_smoothed)
print(log_probability_smoothed) 

-7.215239978730098


## Phase 8: Next word prediction

In [84]:
# prefix will be used in predict_next word
def tokenize_prefix(sentence):
    puncs = ".?!,"
    sentence = sentence.lower()
    for punc in puncs:
        sentence = sentence.replace(punc,'')
    tokenized = sentence.split()

    return ["<s>"] + tokenized

In [85]:
prefix = tokenize_prefix("I love")
prefix[-1]

'love'

In [86]:
def predict_next_word(tokenized_prefix: list, bigram_probabilities):
    max_value = 0
    predicted_word = None # IF the last word in the prefix was never exist as a previous word the function will return None 
    last_word = tokenized_prefix[-1]

    for (previous_word, current_word), value in  bigram_probabilities.items():
        if previous_word == last_word and value > max_value:
            max_value = value
            predicted_word = current_word
        
    return predicted_word, max_value

In [87]:
print(predict_next_word(prefix,bigram_probabilities ))

('machine', 0.5)


In [88]:
bigram_probabilities = {
    ("love", "machine"): 0.40,
    ("love", "python"): 0.25,
    ("love", "learning"): 0.25,
    
    ("love", "cats"): 0.05,
    ("love", "pizza"): 0.25,
    ("machine", "learning"): 0.70,
    ("machine", "is"): 0.30,

    ("i", "love"): 0.80,

}

tokenized_prefix = ["<s>", "i", "love"]

In [89]:
def predict_top_k(tokenized_prefix: list, bigram_probabilities, top_k=5):
    
    
    ranked_predictions = {} 
    top_values = []
    top_words = []
    last_word = tokenized_prefix[-1]

    for (previous_word, current_word), value in  bigram_probabilities.items():
        if previous_word == last_word :            
            top_values.append(value)
            top_words.append(current_word)

    predictions =[(top_words,top_values) for top_values,top_words in sorted(zip(top_values,top_words), reverse=True,)] # List of tuples are hashables
    ranked_predictions = dict(predictions[0:top_k])


    return ranked_predictions

In [90]:
predicted_top_k = predict_top_k(tokenized_prefix, bigram_probabilities, 3)
predicted_top_k

{'machine': 0.4, 'python': 0.25, 'pizza': 0.25}

## Phase 9: Perplexity

In [91]:
def compute_perplexity(tokenized_sentence, bigram_probabilities):
    log_probability = compute_sentence_log_probability(tokenized_sentence, bigram_probabilities)
    predictions_number = len(tokenized_sentence) - 1
    perplexity = np.exp(-(log_probability/predictions_number))
    return perplexity

In [92]:
sentence =  "I love machine learning"
tokenized_sentence = tokenize_sentence(sentence)
tokenized_sentence

['<s>', 'i', 'love', 'machine', 'learning', '</s>']

In [93]:
perplexity = compute_perplexity(sentence, bigram_probabilities)

In [94]:
perplexity 


np.float64(inf)